# Pandas 必知必会速查

建一个简单的 DataFrame 用来演示所有操作：

In [9]:
import pandas as pd

# 手动造一个 DataFrame
df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六", "钱七"],
    "age": [20, 22, None, 17, 23],           # None 就是缺失值
    "score": [85, 90, 78, 92, None],
    "city": ["北京", "上海", "北京", "广州", "上海"]
},index=[1,2,3,4,5],columns=["name","age","score","city"])
df

,name,age,score,city
1,张三,20.0,85.0,北京
2,李四,22.0,90.0,上海
3,王五,NaN,78.0,北京
4,赵六,17.0,92.0,广州
5,钱七,23.0,NaN,上海


---
## 1. 读取和写出数据

```python
pd.read_csv("xxx.csv")        # CSV → DataFrame
pd.read_json("xxx.json")      # JSON → DataFrame
df.to_csv("xxx.csv", index=False)  # DataFrame → CSV，index=False 不写行号

# 常用参数
pd.read_csv("file.csv", encoding="utf-8")        # 指定编码
pd.read_csv("file.csv", usecols=["name", "age"])  # 只读指定列
pd.read_csv("file.csv", nrows=100)                # 只读前 100 行
```

---
## 2. 数据预览

```python
df.head()       # 前 5 行
df.head(10)     # 前 10 行
df.tail()       # 后 5 行
df.info()       # 每列的类型、非空数量、内存占用（不返回数据，打印摘要）
df.describe()   # 数值列的统计：个数、均值、标准差、min、25%、50%、75%、max
df.shape        # 返回 (行数, 列数)，注意没有括号，是属性不是方法
df.columns      # 返回所有列名
df.dtypes       # 每列的数据类型
```

In [10]:
# head / tail
df.head(2)

,name,age,score,city
1,张三,20.0,85.0,北京
2,李四,22.0,90.0,上海


In [11]:
# info: 看看各列有没有缺失值
df.info()

<class 'pandas.DataFrame'>
Index: 5 entries, 1 to 5
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    5 non-null      str    
 1   age     4 non-null      float64
 2   score   4 non-null      float64
 3   city    5 non-null      str    
dtypes: float64(2), str(2)
memory usage: 200.0 bytes


In [12]:
# describe: 数值列的统计摘要
df.describe()

,age,score
count,4.000000,4.000000
mean,20.500000,86.250000
std,2.645751,6.238322
min,17.000000,78.000000
25%,19.250000,83.250000
50%,21.000000,87.500000
75%,22.250000,90.500000
max,23.000000,92.000000


In [13]:
# shape / columns / dtypes
print(df.shape)     # (5, 4) → 5 行 4 列
print(df.columns)   # Index(['name', 'age', 'score', 'city'])
print(df.dtypes)    # name object, age float64, ...

(5, 4)
Index(['name', 'age', 'score', 'city'], dtype='str')
name         str
age      float64
score    float64
city         str
dtype: object


---
## 3. 列操作

```python
df["column"]             # 取一列 → Series
df[["col1", "col2"]]     # 取多列 → DataFrame（注意两层方括号！）
df["new_col"] = ...      # 新增列 / 修改已有列
df.drop("col", axis=1)   # 删除列（axis=1 表示列方向）
df.rename(columns={"old": "new"})  # 重命名列
```

In [14]:
# 取一列
df["name"]

1    张三
2    李四
3    王五
4    赵六
5    钱七
Name: name, dtype: str

In [15]:
# 取多列（注意 [[]]）
df[["name", "score"]]

,name,score
1,张三,85.0
2,李四,90.0
3,王五,78.0
4,赵六,92.0
5,钱七,NaN


In [16]:
# 新增一列
df["level"] = ["A", "A", "B", "A", "B"]
df

,name,age,score,city,level
1,张三,20.0,85.0,北京,A
2,李四,22.0,90.0,上海,A
3,王五,NaN,78.0,北京,B
4,赵六,17.0,92.0,广州,A
5,钱七,23.0,NaN,上海,B


---
## 4. 筛选

```python
df[df["age"] > 18]              # 布尔索引：df[条件]
df.query("age > 18")             # query 写法：括号里是字符串
df.query("age > 18 and city == '上海'")  # 多条件

# 多个条件（不能用 and/or，要用 & / |）
df[(df["age"] > 18) & (df["city"] == "上海")]   # & 不是 and
df[(df["age"] < 18) | (df["score"] > 90)]         # | 不是 or

# 常用筛选方式
df["age"].isin([20, 22])         # 值在列表中
df["age"].between(18, 25)        # 在区间内
df["name"].str.contains("张")     # 字符串包含（.str 访问器）
```

In [17]:
# 布尔索引
df[df["age"] > 20]

,name,age,score,city,level
2,李四,22.0,90.0,上海,A
5,钱七,23.0,NaN,上海,B


In [18]:
# query 写法
df.query("age > 20 and city == '上海'")

,name,age,score,city,level
2,李四,22.0,90.0,上海,A
5,钱七,23.0,NaN,上海,B


---
## 5. 处理缺失值

```python
df.isnull()        # 每个单元格是不是缺失 → True/False 表
df.isnull().sum()  # 每列缺失数量（True=1, False=0）
df.notnull()       # 反过来，不是缺失的

df.dropna()               # 删掉有缺失值的行
df.dropna(subset=["age"]) # 只删 age 列缺失的行
df.dropna(how="all")      # 只有整行全空才删

df.fillna(0)              # 所有缺失值填 0
df.fillna({"age": 0, "score": 60})  # 不同列填不同值
df["age"].fillna(df["age"].mean())  # 用均值填充（常用！）
```

In [19]:
# 检查缺失
df.isnull()

,name,age,score,city,level
1,False,False,False,False,False
2,False,False,False,False,False
3,False,True,False,False,False
4,False,False,False,False,False
5,False,False,True,False,False


In [ ]:
# 每列缺失数量
df.isnull().sum()

name     0
age      1
score    1
city     0
level    0
dtype: int64

In [ ]:
# 删掉有缺失的行
df.dropna(subset=["age", "score"])      #只要 age 或 score 中任意一个是 NaN，该行就会被删除

,name,age,score,city,level
1,张三,20.0,85.0,北京,A
2,李四,22.0,90.0,上海,A
4,赵六,17.0,92.0,广州,A


In [22]:
# 用均值填充（注意原 df 没变，要赋值回去）
df["age"] = df["age"].fillna(df["age"].mean())
df["score"] = df["score"].fillna(60)
df

,name,age,score,city,level
1,张三,20.0,85.0,北京,A
2,李四,22.0,90.0,上海,A
3,王五,20.5,78.0,北京,B
4,赵六,17.0,92.0,广州,A
5,钱七,23.0,60.0,上海,B


---
## 6. 分组聚合 groupby

```python
# 套路：df.groupby("分组列")["要算的列"].聚合函数()
df.groupby("city")["score"].mean()       # 按城市算平均分
df.groupby("city")["age"].max()          # 按城市算最大年龄
df.groupby("city")["score"].agg(["mean", "count"])  # 同时算多个指标
df.groupby(["level", "city"])["score"].mean()         # 多级分组

# 常用聚合函数：sum、mean、max、min、count、std、median
```

In [34]:
# 按城市分组，算平均分
df.groupby("city")["score"].mean()

city
上海    75.0
北京    81.5
广州    92.0
Name: score, dtype: float64

In [24]:
# 同时算多个指标
df.groupby("level")["score"].agg(["mean", "count"])

,mean,count
level,,
A,89.0,3
B,69.0,2


level  city
A      上海      90.0
       北京      85.0
       广州      92.0
B      上海      60.0
       北京      78.0
Name: score, dtype: float64

---
## 7. apply — 对每列/每行应用函数

```python
# 对一列用
df["score"].apply(lambda x: x * 2)         # 每个 score 翻倍
df["city"].apply(lambda x: x.upper())       # 城市名全大写

# 对整行用（axis=1）
df.apply(lambda row: row["age"] + row["score"], axis=1)  # 每行的 age+score

# apply 适合：已有的函数没法直接对 DataFrame 用的时候
df["city"].apply(len)  # 每个城市名的长度
```

In [25]:
# 每个分数加 10
df["score"].apply(lambda x: x + 10)

1     95.0
2    100.0
3     88.0
4    102.0
5     70.0
Name: score, dtype: float64

In [26]:
# 每行的 age + score
df.apply(lambda row: row["age"] + row["score"], axis=1)

1    105.0
2    112.0
3     98.5
4    109.0
5     83.0
dtype: float64

---
## 8. merge — 两个 DataFrame 的合并（JOIN）

```python
# 内连接：只保留两边都匹配的行（默认 how="inner"）
pd.merge(df1, df2, on="key")

# 左连接：保留左边所有行，右边没匹配的填空
pd.merge(df1, df2, on="key", how="left")

# 外连接：保留两边所有行
pd.merge(df1, df2, on="key", how="outer")

# 左右 key 名不同
pd.merge(df1, df2, left_on="city", right_on="城市")
```

| how | 相当于 | 行为 |
|-----|--------|------|
| inner（默认）| SQL INNER JOIN | 只保留匹配的 |
| left | SQL LEFT JOIN | 左边全保留 |
| right | SQL RIGHT JOIN | 右边全保留 |
| outer | SQL FULL OUTER JOIN | 两边都保留 |

In [27]:
# 造两个表来演示 merge
cities = pd.DataFrame({
    "city": ["北京", "上海", "广州"],
    "region": ["华北", "华东", "华南"]
})
cities

,city,region
0,北京,华北
1,上海,华东
2,广州,华南


In [28]:
# 内连接：给原始数据补上地区信息
pd.merge(df, cities, on="city", how="inner")

,name,age,score,city,level,region
0,张三,20.0,85.0,北京,A,华北
1,李四,22.0,90.0,上海,A,华东
2,王五,20.5,78.0,北京,B,华北
3,赵六,17.0,92.0,广州,A,华南
4,钱七,23.0,60.0,上海,B,华东


In [29]:
# 左连接：保留 df 所有行，cities 里没有的就空着
pd.merge(df, cities, on="city", how="left")

,name,age,score,city,level,region
0,张三,20.0,85.0,北京,A,华北
1,李四,22.0,90.0,上海,A,华东
2,王五,20.5,78.0,北京,B,华北
3,赵六,17.0,92.0,广州,A,华南
4,钱七,23.0,60.0,上海,B,华东


---
## 记忆要点

| 场景 | 操作 |
|------|------|
| 拿到数据第一步 | `df.head()`, `df.info()`, `df.shape` |
| 挑数据 | `df[["a","b"]]`, `df[df["age"]>20]` |
| 改数据 | `df["new"] = ...`, `df["x"].apply(...)` |
| 有空值 | `df.isnull().sum()`, `df.fillna(0)` |
| 分类统计 | `df.groupby("x")["y"].mean()` |
| 拼表 | `pd.merge(a, b, on="key")` |
| 存结果 | `df.to_csv("out.csv", index=False)` |